In [1]:
from Scene_lib2 import *
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import pickle

In [ ]:
# Define parameters
ns = np.arange(start=50, stop=1501, step=50)
categories = ['Measured']
noise_levels = [0.]
base_dir = '3D-Data'
relative_dir = "Measured_relative"  # New folder for relative data
h = 3.5
boundary = [[-4,4],[-4,4],[0,h]]

# Generate 10 datasets with different seeds
num_seeds = 10

for category in categories:
    for noise_level in noise_levels:
        for n in ns:
            for seed in range(num_seeds):  # Iterate over 10 seeds (0 to 9)
                my_scene = Scene()
                use_measured_data = (category == 'Measured')

                my_scene.generate_sven_lights(use_measured_data=use_measured_data)
                my_scene.generate_diode_plane(n=2)
                my_scene.generate_diode_plane(n=2, h=h)
                my_scene.generate_diode_halton_volume(n=n-8, volume_boundary=[[-4,4],[-4,4],[0,h]], seed=seed)

                # Generate relative RSS dataset
                data = my_scene.generate_rss_table_with_diode_coords_relative(noise_std=noise_level, seed=seed)
                print(data)

                # Create category folder if it doesn’t exist
                category_dir = os.path.join(base_dir, relative_dir)
                os.makedirs(category_dir, exist_ok=True)

                # Save the dataset with seed info in filename
                file_name = f"relative_rss_n={n}_noise={noise_level}_seed={seed}.csv"
                file_path = os.path.join(category_dir, file_name)
                
                data.to_csv(file_path, index=False)
                print(f"Saved relative RSS data for {category} (n={n}, noise={noise_level}, seed={seed}) to {file_path}")


         RSS0/RSS1  RSS0/RSS2  RSS0/RSS3  RSS0/RSS4  RSS1/RSS2  RSS1/RSS3  \
Diode0    0.346366   1.252824   1.293881   2.913172   3.617053   3.735591   
Diode1    1.313671   2.862727   0.342459   1.286464   2.179181   0.260688   
Diode2    1.295934   0.354254   3.071588   1.291836   0.273358   2.370173   
Diode3    2.982246   1.309477   1.335298   0.346099   0.439091   0.447749   
Diode4    0.044505   1.257554   1.796955  17.969563  28.256765  40.376893   
Diode5    2.195829  14.424479   0.035065   1.919944   6.569036   0.015969   
Diode6    2.199755   0.035963  45.599191   1.764854   0.016349  20.729216   
Diode7   23.189557   2.523751   9.623246   0.032844   0.108831   0.414982   
Diode8    0.346366   1.252824   1.293881   2.913172   3.617053   3.735591   
Diode9    1.261976   2.554409   1.286932    2.63217   2.024135   1.019775   
Diode10   1.611335   0.518554   4.587571   2.388544   0.321817   2.847062   
Diode11   1.376676   5.421762   0.197099   2.484493   3.938299    0.14317   

In [ ]:
import os
import numpy as np

# Define parameters
ns = np.array([50])  # Dense grid setting
categories = ['Measured']
noise_levels = [0., 0.000006, 0.000019, 0.00006, 0.00019]
base_dir = '3D-Data'
relative_dir = "Measured_relative"  # New folder for relative test data
h = 3.5  # Height boundary

# Iterate over all categories, noise levels, and grid sizes
for category in categories:
    for noise_level in noise_levels:
        for n in ns:
            my_scene = Scene()
            use_measured_data = (category == 'Measured')

            my_scene.generate_sven_lights(use_measured_data=use_measured_data)
            my_scene.generate_diode_volume(n=n, volume_boundary=[[-4,4],[-4,4],[0,h]])

            # Generate relative RSS dataset
            data = my_scene.generate_rss_table_with_diode_coords_relative(noise_std=noise_level, seed=0)
            print(data)

            # Create category folder if it doesn’t exist
            category_dir = os.path.join(base_dir, relative_dir)
            os.makedirs(category_dir, exist_ok=True)

            # Save the dataset with noise level info in filename
            file_name = f"relative_gnd_n={n}_noise={noise_level}.csv"
            file_path = os.path.join(category_dir, file_name)

            data.to_csv(file_path, index=False)
            print(f"Saved relative RSS test data for {category} (n={n}, noise={noise_level}) to {file_path}")


             RSS0/RSS1 RSS0/RSS2 RSS0/RSS3 RSS0/RSS4 RSS1/RSS2 RSS1/RSS3  \
Diode0        0.346366  1.252824  1.293881  2.913172  3.617053  3.735591   
Diode1         0.34009  1.257451  1.298513  2.951419  3.697413  3.818151   
Diode2        0.333774  1.261945  1.303292  2.991484  3.780832  3.904711   
Diode3        0.327452  1.266482  1.308122  3.033395  3.867683  3.994847   
Diode4        0.321106  1.271482  1.313476  3.076186   3.95969   4.09047   
...                ...       ...       ...       ...       ...       ...   
Diode124995  15.421068  2.122734  5.250573   0.05347  0.137652   0.34048   
Diode124996  17.054439  2.200797  6.242326  0.048079  0.129045  0.366024   
Diode124997  19.531022  2.287246  7.316661  0.042818  0.117108  0.374617   
Diode124998  20.894791  2.407865  8.828673  0.037756  0.115238   0.42253   
Diode124999  23.189557  2.523751  9.623246  0.032844  0.108831  0.414982   

            RSS1/RSS4 RSS2/RSS3 RSS2/RSS4 RSS3/RSS4    X    Y         Z  
Diode0       

In [4]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Define paths
base_dir = "3D-Data"
category = "Measured_relative"
normalized_category = "Measured_relative_Normalized"
scaler_filename = "scaler_relative.pkl"

# Ensure the normalized folder exists
os.makedirs(os.path.join(base_dir, normalized_category), exist_ok=True)

# Select a reference file to compute the scaler
n = 50
noise_level = 0.0
file_name = f"relative_gnd_n={n}_noise={noise_level}.csv"
file_path = os.path.join(base_dir, category, file_name)

# Load data
data = pd.read_csv(file_path)
columns = data.columns  # Store original column names for later

# Separate RSS ratio columns and coordinate columns
rss_ratio_columns = [col for col in columns if "/" in col]  # Detect RSS ratio columns
coordinate_columns = ["X", "Y", "Z"]

# Apply logarithm transformation only to RSS ratio columns
data_log = data.copy()
data_log[rss_ratio_columns] = np.log(data[rss_ratio_columns])

# Generate and save the scaler
regenerate_scaler = True
if regenerate_scaler:
    scaler = StandardScaler()
    scaler.fit(data_log.values)  # Fit on log-transformed data
    with open(scaler_filename, "wb") as f:
        pickle.dump(scaler, f)
    print(f"Scaler saved as '{scaler_filename}'")
else:
    with open(scaler_filename, "rb") as f:
        scaler = pickle.load(f)

# Transform the data
scaled_data = pd.DataFrame(scaler.transform(data_log.values), columns=columns)

# Save the normalized data
scaled_file_path = os.path.join(base_dir, normalized_category, file_name)
scaled_data.to_csv(scaled_file_path, index=False)
print(f"Scaled relative data saved as '{scaled_file_path}'")


Scaler saved as 'scaler_relative.pkl'
Scaled relative data saved as '3D-Data\Measured_relative_Normalized\relative_gnd_n=50_noise=0.0.csv'
